## Phần 0 — Khởi động (5 phút)

- [ ] **0.1 Seed và device.** Viết hàm `seed_everything(seed)` đặt seed cho `random`, `numpy`, `torch` (CPU + CUDA). Tạo biến `device` tự chọn `cuda` nếu có, không thì `cpu`.
  - **Kiểm tra:** gọi `seed_everything(42)` hai lần, mỗi lần in `torch.rand(3)` → hai kết quả giống hệt nhau.

<details><summary>Gợi ý cú pháp</summary>

`random.seed`, `np.random.seed`, `torch.manual_seed`, `torch.cuda.manual_seed_all`, `torch.cuda.is_available()`

In [167]:
import numpy as np
import torch
import random
import os
import pandas as pd

In [168]:
def seed_everything(seed: int):
    ''' seed python before
    seed random lib in python
    seed numpy 
    seed torch
    seed cuda 
    seed compute in GPU
    ** all things in python - becuz it is independence
    =====================
    params: seed (int)
    return device - seed '''

    os.environ['PYTHONASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.cuda.seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(torch.rand((5, 4)))
    print(device)
    return device
    

In [169]:
seed_everything(42)

tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274]])
mps


device(type='mps')

**1.1 Đọc nhãn.** Đọc `train.csv` bằng pandas. In số ảnh, số lớp, số ảnh mỗi lớp. Tạo `class2idx` và `idx2class`.

- **Kiểm tra:** `len(class2idx) == 6`; `idx2class[class2idx['glass']] == 'glass'`.

In [170]:
train_dir = '/Users/quangmanh/Project/Final-competitive/Olympic-AI-Practice-1/đề code tay/data/data-2/TACVU2/train/train.csv'
df = pd.read_csv(train_dir)
print(df.shape)
df.head()

(10612, 2)


,file_name,category_id
0,000001.jpg,5
1,000002.jpg,2
2,000003.jpg,1
3,000004.jpg,2
4,000005.jpg,2


In [171]:
num_image = df.shape[0]
num_class = np.sort(df['category_id'].unique()).tolist()
num_class

[1, 2, 3, 4, 5, 6]

In [172]:
df['category_id'].value_counts()

category_id
4    2492
2    2107
5    2023
3    1722
1    1694
6     574
Name: count, dtype: int64

In [173]:
df_count = df['category_id'].value_counts().reset_index()
df_count.columns = ['category_id', 'image_count']
df_count

,category_id,image_count
0,4,2492
1,2,2107
2,5,2023
3,3,1722
4,1,1694
5,6,574


In [183]:
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
class_to_idx = {idx: name for idx, name in enumerate(CLASSES)}
class_to_idx

{0: 'cardboard', 1: 'glass', 2: 'metal', 3: 'paper', 4: 'plastic', 5: 'trash'}

In [175]:
def class2idx(df):
    name_atr = list(df.columns)[-2]
    return np.sort(df[name_atr].unique(), kind='quicksort').tolist()
    

In [188]:
class_to_idx[0]

'cardboard'

In [198]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10612 entries, 0 to 10611
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   file_name  10612 non-null  object
 1   label      10612 non-null  int64 
 2   class      10612 non-null  object
dtypes: int64(1), object(2)
memory usage: 248.8+ KB


In [201]:
def idx2class(df):
    columns = df.columns
    return df[df['class'] == class_to_idx[5]]

In [204]:
idx2class(df).count()

file_name    574
label        574
class        574
dtype: int64

In [185]:
df.head()

,file_name,label,class
0,000001.jpg,4,plastic
1,000002.jpg,1,glass
2,000003.jpg,0,cardboard
3,000004.jpg,1,glass
4,000005.jpg,1,glass


In [178]:
df['label'] = df['category_id'] - 1

In [179]:
df = df.drop(columns=['category_id'])

In [184]:
df['class'] = df['label'].map(class_to_idx)

True